In [125]:
import numpy as np
from dataclasses import dataclass, make_dataclass
import pandas as pd
from sklearn.datasets import load_iris, load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from geneticengine.grammar.grammar import extract_grammar
from geneticengine.algorithms.gp.gp import GeneticProgramming
from geneticengine.algorithms.gp.gp import GeneticProgramming
from geneticengine.evaluation.tracker import ProgressTracker
from geneticengine.evaluation.sequential import SequentialEvaluator
from geneticengine.grammar.grammar import extract_grammar
from geneticengine.problems import SingleObjectiveProblem
from geneticengine.representations.tree.initializations import MaxDepthDecider
from geneticengine.random.sources import NativeRandomSource
from geneticengine.representations.tree.treebased import TreeBasedRepresentation
from geneticengine.evaluation.recorder import SearchRecorder
from geneticengine.solutions.individual import Individual
from geneticengine.evaluation.budget import EvaluationBudget
from geneticengine.algorithms.gp.operators.combinators import ParallelStep, SequenceStep
from geneticengine.algorithms.gp.operators.crossover import GenericCrossoverStep
from geneticengine.algorithms.gp.operators.mutation import GenericMutationStep
from geneticengine.algorithms.gp.operators.elitism import ElitismStep
from geneticengine.algorithms.gp.operators.novelty import NoveltyStep
from geneticengine.algorithms.gp.operators.selection import TournamentSelection

In [126]:
wine = load_wine()
X, y = wine.data, wine.target

In [127]:
n_features = X.shape[1]

fields = [(f'f{i}', bool) for i in range(n_features)]

FeatureMask = make_dataclass('FeatureMask', fields)

grammar = extract_grammar([FeatureMask], FeatureMask)

In [128]:
def fitness_function(mask: FeatureMask) -> float:
    selected_features = [getattr(mask, f'f{i}') for i in range(n_features)]
    if not any(selected_features):
        return 0.0 #useless
    X_subset = X[:, selected_features]

    clf = LogisticRegression(random_state=0, max_iter=200, solver='liblinear')
    f1 = cross_val_score(clf, X_subset, y, cv=3, scoring='f1_macro').mean()

    return f1

In [129]:
class GenerationRecorder:
    def __init__(self):
        self.rows = []
    def register(self, tracker, individual, problem, is_best):
        self.rows.append({
            "execution_time": tracker.get_elapsed_time(),
            "generation": individual.metadata.get("generation"),
            "fitness": individual.get_fitness(problem).fitness_components[0],
            "phenotype": str(individual.phenotype)
        })

In [130]:
rnd = NativeRandomSource(122)
decider = MaxDepthDecider(rnd, grammar, max_depth=5)
representation = TreeBasedRepresentation(grammar, decider) 

In [131]:
mutation_p = 0.3
crossover_p = 0.9

In [132]:
step = ParallelStep(
    steps = [
        ElitismStep(),
        NoveltyStep(),
        SequenceStep(
            TournamentSelection(tournament_size=2),
            GenericCrossoverStep(probability=crossover_p),
            GenericCrossoverStep(probability=mutation_p),
        ),
    ],
    weights=[0.05,0.05, 0.9]
)

In [133]:
recorder = GenerationRecorder()

In [134]:
tracker = ProgressTracker(
    problem=SingleObjectiveProblem(fitness_function=fitness_function, minimize=False),
    evaluator=SequentialEvaluator(),
    recorders=[recorder]
)

In [135]:
gp = GeneticProgramming(
    problem=tracker.problem,
    budget=EvaluationBudget(10000),
    representation=representation,
    random=rnd,
    tracker=tracker,
    population_size=50,
    step=step,
)

In [136]:
best = gp.search()

In [137]:
best_individual = best[0]
best_f1 = best_individual.get_fitness(gp.get_problem())
print(best_f1, best_individual.get_phenotype())

[0.9455275063990038] FeatureMask(f0=True, f1=True, f2=False, f3=True, f4=True, f5=False, f6=True, f7=True, f8=True, f9=True, f10=False, f11=True, f12=True)


In [138]:
df = pd.DataFrame(recorder.rows)
df.to_csv("../gp_outputs/full_gp_generation.csv", index=False)